# Lab: shortest latency paths between continents
How fast can a LEO constellation carry a packet from **Paris** to **New York**, or all the way to **Sydney**? In this lab you will:
1. build a denser version of the **IRIS²-like** constellation of the previous labs: Walker Delta, **12 orbital planes × 22 satellites** (264 satellites), 1 200 km, with ground stations in **Paris**, **New York** and **Sydney**
2. maintain a **graph of the network in real time** from the simulation events
3. at every tick, compute the **shortest path** (by propagation latency) from Paris, the source, to two destinations: New York and Sydney
4. replay it in an **animated Cesium page**: both paths are underlined on the globe, each in its own color, and a **latency chart** follows the clock

It reuses the event factory of [inter_shell_cesium_lab.ipynb](inter_shell_cesium_lab.ipynb) and the real-time graph [`dynamic_topology_graph.py`](dynamic_topology_graph.py).

In [ ]:
import math
from concurrent.futures import ThreadPoolExecutor

import networkx as nx
from geopy.distance import great_circle

from sat_com_application.event_bus.bus import EventBus
from sat_com_application.event_bus.events import (
    EventType,
    ISLCreatedEvent,
    ISLUpdatedEvent,
    ISLRemovedEvent,
    GSLCreatedEvent,
    GSLRemovedEvent,
)
from sat_com_application.event_bus.factories import DefaultISLEventFactory, EventFactories
from sat_com_builder.configuration_manager import BaseConfigurationManager
from sat_com_builder.models import SimulationProperty
from sat_com_model.models import InterSatelliteLink

from cesium_graph_visualizer import AnimatedCesiumGraphVisualizer
from dynamic_topology_graph import DynamicTopologyGraph, SPEED_OF_LIGHT_KM_S

## 1. The constellation and three ground stations
The IRIS²-like shell keeps its 4 optical terminals (forward, backward, left, right) and its full-mesh addressing plan. Its 55° inclination covers Sydney (34° S) as well as Paris (49° N).

The three ground stations share one group, written inline. `shells_whitelist` restricts them to the IRIS² shell, and each one keeps a single satellite at a time (`best-angle-until-disconnection`, `maximum_connected_satellites: 1`). With a single link, a ground station can never be a relay in the middle of a path: every path goes through space.

We simulate 2 hours, one tick every 30 seconds.

In [ ]:
def optical_terminal(azimuth, descriptor):
    return {
        "azimuth_angle_configuration": azimuth,
        "elevation_angle_configuration": -90.0,
        "descriptor": descriptor,
        "orientation_possibilities": [descriptor],
    }


CITIES = {
    "Paris": (48.8566, 2.3522),
    "New York": (40.7128, -74.0060),
    "Sydney": (-33.8688, 151.2093),
}

lab_config = {
    "simulation_name": "Shortest Latency Paths Lab",
    "start_date": "2026-01-01 00:00:00.000000",
    "end_date": "2026-01-01 02:00:00.000000",
    "movement_model": "pyorbital",
    "distance_model": "sklearn",
    "ground_objects_properties": [
        {
            "identifier": "City Gateways",
            "type": "ground_station",
            "ground_objects": [
                {"id": index, "label": city, "latitude": latitude, "longitude": longitude, "altitude": 50.0}
                for index, (city, (latitude, longitude)) in enumerate(CITIES.items())
            ],
            "connectivity_properties": {
                "elevation_above_horizon": 20,
                "shells_whitelist": ["IRIS2"],
                "maximum_connected_satellites": 1,
                "ground_to_space_connections_strategy": "best-angle-until-disconnection",
            },
        }
    ],
    "walker_shells": [
        {
            "type": "delta",
            "static_addressing_plan": "FULL_MESH_WALKER_DELTA",
            "satellite_property": {
                "optical_inter_satellite_link_installations": [
                    optical_terminal(0.0, "INTRA_PLANE_FORWARD"),
                    optical_terminal(-180.0, "INTRA_PLANE_BACKWARD"),
                    optical_terminal(90.0, "INTER_PLANE_LEFT"),
                    optical_terminal(-90.0, "INTER_PLANE_RIGHT"),
                ]
            },
            "constellation_property": {
                "identifier": "IRIS2",
                "amount_of_orbit_plane": 12,
                "amount_of_satellite_per_orbit_plane": 22,
                "inclination": 55.0,
                "phase_difference_between_satellites": True,
                "mean_revolution_per_day": 13.16,
            },
            "orbital_connectivity_property": {"adjacent_inter_satellite_shifting": 0},
        }
    ],
}

simulation_properties = SimulationProperty(**lab_config)

## 2. Satellite positions in the ISL events
Same event factory as in [inter_shell_cesium_lab.ipynb](inter_shell_cesium_lab.ipynb): it adds the position, name and shell of both satellites to `ISL_CREATED` / `ISL_UPDATED`, so the events alone tell where every satellite is at every tick.

In [ ]:
class PositionedISLCreatedEvent(ISLCreatedEvent):
    satellite_a_name: str
    satellite_b_name: str
    satellite_a_shell: str
    satellite_b_shell: str
    satellite_a_position: tuple[float, float, float]  # longitude, latitude, altitude (m)
    satellite_b_position: tuple[float, float, float]


class PositionedISLUpdatedEvent(ISLUpdatedEvent):
    satellite_a_name: str
    satellite_b_name: str
    satellite_a_shell: str
    satellite_b_shell: str
    satellite_a_position: tuple[float, float, float]
    satellite_b_position: tuple[float, float, float]


def position_of(topology_object):
    coordinates = topology_object.get_coordinates()
    return coordinates.longitude, coordinates.latitude, coordinates.altitude


class PositionedISLEventFactory(DefaultISLEventFactory):

    @staticmethod
    def _satellites_information(link: InterSatelliteLink) -> dict:
        satellite_a = link.get_satellite_source()
        satellite_b = link.get_satellite_destination()
        return dict(
            satellite_a_name=satellite_a.satellite_name,
            satellite_b_name=satellite_b.satellite_name,
            satellite_a_shell=satellite_a.shell.identifier,
            satellite_b_shell=satellite_b.shell.identifier,
            satellite_a_position=position_of(satellite_a),
            satellite_b_position=position_of(satellite_b),
        )

    def created(self, link: InterSatelliteLink) -> PositionedISLCreatedEvent:
        event = super().created(link)
        return PositionedISLCreatedEvent(**event.model_dump(), **self._satellites_information(link))

    def updated(self, link: InterSatelliteLink) -> PositionedISLUpdatedEvent:
        event = super().updated(link)
        return PositionedISLUpdatedEvent(**event.model_dump(), **self._satellites_information(link))


event_factories = EventFactories(isl_event_factory=PositionedISLEventFactory())

## 3. Listen to the events
Two consumers share the same event bus:
- our handlers record the satellite positions and the lifetime of every link (ISL and ground link), for the Cesium replay
- the `DynamicTopologyGraph` keeps an igraph graph in sync with the network: one vertex per object, one edge per link, weighted by the link `distance_km`, refreshed by every `*_UPDATED` event

In [ ]:
satellites = {}
open_links = {}
link_lifetimes = {}


def record_positions(event):
    for satellite_id, name, shell, position in (
        (event.satellite_a_id, event.satellite_a_name, event.satellite_a_shell, event.satellite_a_position),
        (event.satellite_b_id, event.satellite_b_name, event.satellite_b_shell, event.satellite_b_position),
    ):
        satellite = satellites.setdefault(satellite_id, {"name": name, "shell": shell, "positions": {}})
        satellite["positions"][event.sim_time] = position


def open_link(link_id, object_a_id, object_b_id, direction, start_time):
    open_links[link_id] = (object_a_id, object_b_id, direction, start_time)


def close_link(link_id, end_time):
    object_a_id, object_b_id, direction, start_time = open_links.pop(link_id)
    pair = tuple(sorted((object_a_id, object_b_id)))
    lifetime = link_lifetimes.setdefault(pair, {"direction": direction, "intervals": []})
    lifetime["intervals"].append((start_time, end_time))


async def on_isl_created(event: PositionedISLCreatedEvent):
    record_positions(event)
    open_link(event.link_id, event.satellite_a_id, event.satellite_b_id, event.direction, event.sim_time)


async def on_isl_updated(event: PositionedISLUpdatedEvent):
    record_positions(event)


async def on_link_removed(event):
    close_link(event.link_id, event.sim_time)


async def on_gsl_created(event: GSLCreatedEvent):
    open_link(event.link_id, event.ground_station_id, event.satellite_id, "ground", event.sim_time)


event_bus = EventBus()
event_bus.subscribe(EventType.ISL_CREATED, on_isl_created)
event_bus.subscribe(EventType.ISL_UPDATED, on_isl_updated)
event_bus.subscribe(EventType.ISL_REMOVED, on_link_removed)
event_bus.subscribe(EventType.GSL_CREATED, on_gsl_created)
event_bus.subscribe(EventType.GSL_REMOVED, on_link_removed)

topology_graph = DynamicTopologyGraph().attach(event_bus)

## 4. Load the simulation
`run_synchronously` runs the simulation outside Jupyter's event loop, so each handler runs as soon as its event is published (see the note in [optical_terminal_lab.ipynb](optical_terminal_lab.ipynb)).

In [ ]:
def run_synchronously(function, *args, **kwargs):
    """Run `function` outside Jupyter's event loop, so the event bus calls each handler right away."""
    with ThreadPoolExecutor(max_workers=1) as executor:
        return executor.submit(function, *args, **kwargs).result()


configuration_manager = BaseConfigurationManager(
    simulation_property=simulation_properties,
    event_bus=event_bus,
    event_factories=event_factories,
)
simulation_manager = run_synchronously(configuration_manager.load_simulation)

ground_stations = {ground_station.label: ground_station for ground_station in simulation_manager.get_ground_stations()}
print(f"{len(simulation_manager.get_satellites())} satellites")
print(f"ground stations: {', '.join(f'{label} (id {gs.topology_uniq_id})' for label, gs in ground_stations.items())}")
print(f"graph: {topology_graph.graph.vcount()} vertices, {topology_graph.graph.ecount()} edges")

## 5. The shortest path at every tick
`topology_graph.shortest_path_with_latency_ms(source, target)` runs Dijkstra on the graph, weighted by the link distances, and returns the path (the list of objects it goes through) and its propagation latency: `distance (km) / 299 792.458 × 1000`. Only propagation: no processing, queuing or transmission delay.

`measure_routes` runs at every tick, after the simulation update, so the graph already holds every change of the tick. It records, for each route, the path and its latency. An empty path means no route right now (for instance, no satellite above 20° for one of the ground stations).

As a reference, we also compute the latency of light **in an optical fiber** laid along the great circle between both cities (refractive index ≈ 1.468, so light travels at about 2/3 of *c*). Real submarine cables are longer than the great circle, so this is a lower bound for the terrestrial internet.

In [ ]:
FIBER_REFRACTIVE_INDEX = 1.468
ROUTES = {
    "Paris → New York": ("Paris", "New York", "#2a78d6"),
    "Paris → Sydney": ("Paris", "Sydney", "#eb6834"),
}

route_history = {route: [] for route in ROUTES}


def fiber_latency_ms(city_a, city_b):
    distance_km = great_circle(CITIES[city_a], CITIES[city_b]).km
    return distance_km / (SPEED_OF_LIGHT_KM_S / FIBER_REFRACTIVE_INDEX) * 1000


def describe(uid):
    if uid in satellites:
        return satellites[uid]["name"].replace("IRIS2 ", "S")
    return simulation_manager.get_topology_object_from_uniq_id(uid).label


def measure_routes():
    sim_time = simulation_manager.time_manager.current_time
    for route, (source, target, _) in ROUTES.items():
        path, latency_ms = topology_graph.shortest_path_with_latency_ms(
            ground_stations[source].topology_uniq_id, ground_stations[target].topology_uniq_id
        )
        route_history[route].append({"time": sim_time, "path": path, "latency_ms": latency_ms})

    if sim_time.minute % 10 == 0 and sim_time.second == 0:
        for route in ROUTES:
            sample = route_history[route][-1]
            if sample["path"]:
                hops = len(sample["path"]) - 1
                print(f"[{sim_time:%H:%M}] {route:<18} {sample['latency_ms']:6.2f} ms, {hops:>2} hops: "
                      + " → ".join(describe(uid) for uid in sample["path"]))
            else:
                print(f"[{sim_time:%H:%M}] {route:<18} no path")


for route, (source, target, _) in ROUTES.items():
    print(f"{route}: fiber along the great circle {fiber_latency_ms(source, target):.1f} ms")
print()

measure_routes()  # state right after the load
simulation_manager.time_manager.register_action(measure_routes)

## 6. Run the simulation

In [ ]:
run_synchronously(simulation_manager.time_manager.tick_until_the_end, ticking_time_in_seconds=30)

end_time = simulation_manager.time_manager.current_time
for link_id in list(open_links):
    close_link(link_id, end_time)

## 7. The animated Cesium page
We hand `AnimatedCesiumGraphVisualizer` (see [`cesium_graph_visualizer.py`](cesium_graph_visualizer.py)):
- the recorded network, as in the inter-shell lab (satellite positions, link lifetimes, static ground stations)
- `highlighted_paths`: the path of each route at every tick. Each hop is **underlined** on the globe in the route color as long as it is part of the path; the other links are dimmed. Paris → New York is blue, Paris → Sydney is orange: where both routes share hops (around Paris), the wider blue line stays visible under the orange one.
- `time_series`: the latency of both routes, drawn as a **chart** under the globe. Its cursor follows the clock; hover it to read both values, click it to jump to that time. A gap means no path.
- `live_values`: the current latency and hop count of each route, in the legend
- `trail_seconds=0`: no trajectory trail behind the satellites, to keep the paths readable. The orbital planes stay visible anyway: the intra-plane (orbital) ISLs link each satellite to the next one of its plane, so they draw one ring per orbital plane.

Without a [Cesium Ion token](https://ion.cesium.com/tokens?page=1), the page uses the offline Natural Earth imagery shipped with CesiumJS.

In [ ]:
CESIUM_TOKEN = None  # or "<YOUR_CESIUM_TOKEN>"

graph = nx.Graph()
for satellite_id, satellite in satellites.items():
    graph.add_node(
        satellite_id,
        type="Satellite",
        name=satellite["name"],
        shell=satellite["shell"],
        positions=[(time, *position) for time, position in satellite["positions"].items()],
    )
for label, ground_station in ground_stations.items():
    graph.add_node(
        ground_station.topology_uniq_id,
        type="GroundStation",
        name=label,
        shell="Ground Station",
        static=True,
        positions=[(simulation_manager.time_manager.start_date, *position_of(ground_station))],
    )
for (object_a_id, object_b_id), lifetime in link_lifetimes.items():
    graph.add_edge(object_a_id, object_b_id, direction=lifetime["direction"], intervals=lifetime["intervals"])


def route_text(sample):
    if not sample["path"]:
        return "no path"
    return f"{sample['latency_ms']:.2f} ms · {len(sample['path']) - 1} hops"


highlighted_paths = {
    f"{route} path": {
        "color": color,
        "samples": [(sample["time"], sample["path"]) for sample in route_history[route]],
    }
    for route, (_, _, color) in ROUTES.items()
}
time_series = {
    "title": "Propagation latency of the shortest path",
    "unit": "ms",
    "series": {
        route: {"color": color, "samples": [(sample["time"], sample["latency_ms"]) for sample in route_history[route]]}
        for route, (_, _, color) in ROUTES.items()
    },
}
live_values = {
    route: [(sample["time"], route_text(sample)) for sample in route_history[route]] for route in ROUTES
}

visualizer = AnimatedCesiumGraphVisualizer(
    graph,
    start_time=simulation_manager.time_manager.start_date,
    end_time=end_time,
    title="IRIS² (12 × 22): shortest latency paths from Paris",
    cesium_token=CESIUM_TOKEN,
    live_values=live_values,
    highlighted_paths=highlighted_paths,
    time_series=time_series,
    trail_seconds=0,
)
visualizer.generate_html("results/shortest_latency_paths.html")

Open [`results/shortest_latency_paths.html`](results/shortest_latency_paths.html) in your browser. Things to watch:
- both paths always start with the satellite above Paris (a single ground link), then split, sometimes after a few shared hops
- the New York path usually crosses the Atlantic in 4 or 5 hops, close to the fiber latency. But the satellite above Paris does not always have a short chain of links towards New York: the path then takes a long detour (up to about 100 ms), and the blue curve jumps
- the Sydney path is longer and steadier: whatever the detour, it has to cross half the globe
- every few minutes a path **jumps**: a ground station hands over to another satellite, or a shorter chain of satellites appears. The chart shows the same jumps as steps in the latency.

## 8. Recap
For each route: latency range, hop count, how often the path changed, and the comparison with a fiber along the great circle.

In [ ]:
for route, (source, target, _) in ROUTES.items():
    samples = route_history[route]
    reachable = [sample for sample in samples if sample["path"]]
    latencies = [sample["latency_ms"] for sample in reachable]
    hops = [len(sample["path"]) - 1 for sample in reachable]
    path_changes = sum(
        1 for previous, current in zip(samples, samples[1:]) if previous["path"] != current["path"]
    )
    fiber_ms = fiber_latency_ms(source, target)
    faster = sum(1 for latency in latencies if latency < fiber_ms)

    print(route)
    print(f"  reachable    {100 * len(reachable) / len(samples):.0f}% of the ticks")
    if reachable:
        print(f"  latency      min {min(latencies):.2f} ms, mean {sum(latencies) / len(latencies):.2f} ms, max {max(latencies):.2f} ms")
        print(f"  hops         min {min(hops)}, max {max(hops)}")
    print(f"  path changes {path_changes} in 2 hours")
    print(f"  fiber        {fiber_ms:.1f} ms along the great circle: space is faster {100 * faster / len(samples):.0f}% of the ticks")
    print()

## What's next?
- **Add a destination** (for example Paris → Tokyo): add the city to `CITIES`, the route to `ROUTES` with the third slot of the palette, `#1baf7a`.
- **Change the constellation size**: go back to the 10 × 10 shell of the other labs, or grow this one. How does the size change the hops, the latency curve, and how often space beats the fiber?
- **Change the ground strategy**: with `best-angle` (re-evaluated at each tick) instead of `best-angle-until-disconnection`, the ground stations switch satellite more often. Is the latency better?
- Use `topology_graph.hop_counts(...)` to compare the fewest-hops path with the lowest-latency one.